# NL experiment · 05 Table 2 without NL with ResNet-18: the nighttime light of the DHS clusters

This notebook repeats 04, the average nighttime light of the patch of the DHS clusters (the target of 02) from the rows of Table 2 without NL, with ResNet-18 in place of VGG-16. It is 05 of the IWI study with NL moved from the input to the target. Everything else is as in 04: the input bands and their fixed normalisation (GEE scale factors for Landsat, TEMP1 in kelvin / 300), the target, the clusters with complete data and a target value, the spatial folds of 03 (test fold f, validation fold f − 1, training on the other three), Adam with learning rate 1e-3, MSE loss with L2 0.01 on the output weights, batch 64, 100 epochs, and testing the weights with the lowest validation loss.

| key | Table 2 row without NL | input bands |
|---|---|---|
| `base` | Base model using 7 raw Landsat images | 7 Landsat |
| `base_ndvi` | Base model with temporal NDVI | base + 12 monthly NDVI |
| `base_wavelet6` | Base model and 6 statistical wavelet-based features | base + LFC and HFC minimum, maximum and mean |
| `base_hfc_max` | Base model and max values of HFC | base + HFC max |
| `base_hfc_mean` | Base model and mean values of HFC | base + HFC mean |
| `base_hfc_min` | Base model and min values of HFC | base + HFC min |
| `base_lfc_max` | Base model and max values of LFC | base + LFC max |
| `base_lfc_mean` | Base model and mean values of LFC | base + LFC mean |
| `base_lfc_min` | Base model and min values of LFC | base + LFC min |

**Two ways to use ResNet-18** (`RESNET_MODE`, set in Settings or in the environment):

* **`finetune`** (default) is the usual way to train ResNet-18 on satellite bands (Yeh et al. 2020): the first convolution takes all input bands, and every layer learns. Training starts from the ImageNet weights. RED, GREEN and BLUE get the first-layer weights of their ImageNet channel, every other band gets their mean, and all are scaled by 3 / number of bands. Global average pooling gives 512 features for the output layer. `PRETRAINED = False` starts from random weights instead.
* **`frozen`** is the setup of the paper with ResNet-18 in place of VGG-16: a 1 × 1 convolution with ReLU maps the bands to 3 channels, followed by the frozen ImageNet ResNet-18 convolution layers (their BatchNorm keeps the ImageNet statistics), a flatten (7 × 7 × 512, as for VGG-16) and the output layer. Here the network is the only difference from 04.

The two modes write to separate folders, so both can run.

**The target** (`TARGET = 'nl'`, written by 02 and joined to the folds by 03): the mean over the 224 × 224 pixels of the `NL.tif` of the survey year, clipped to 0–150 and normalised with `NormFunc` of the survey year (VIIRS from 2014: (NL − 0.133) / 1.748; DMSP: (NL − 4.153) / 3.476), which is how the IWI study normalised its NL input band. With the default `NL_SENSOR = 'VIIRS'` all clusters are of one period, so the R² is the R² of the patch mean in VIIRS radiance.

**Clusters and folds.** All clusters of `train_clusters_folds_nl.csv` (03: the folds of the IWI study with the target of 02) in VIIRS years (2014 and later, `NL_SENSOR`, as in the paper and in 04/05 of the IWI study), with the spatial folds of 03: fold f is the test fold, the fold before it is the validation fold, and the other three are for training. Clusters with a no-data pixel in any of their three files are left out, as in the IWI study.

**Normalisation.** The Landsat bands get the Collection 2 Level 2 scale factors of the GEE catalog: surface reflectance = DN × 0.0000275 − 0.2, and TEMP1 (ST_B10, ST_B6) = (DN × 0.00341802 + 149) / 300, kelvin divided by 300 as in the TF records. NDVI and wavelet bands are used as they are. No statistics of the data are used. No NL goes in.

**Running.** Run 02 and 03 first. Check `DATA_DIR` and `TORCH_HOME` in Settings, and download the ResNet-18 weights once (section 4; nothing to do if the IWI study has run). The band cache of the IWI study is used as it is, or built first if it is not there. The runs go to `WaveletNL/table2_resnet18_runs_fixednorm/<mode>/`. Each of the 45 runs (9 rows × 5 folds) trains for 100 epochs. Every run saves its state after each epoch, so a stopped job continues where it stopped when you start it again.

**From random weights.** `TRAIN_FROM_SCRATCH=1` in the environment is the same as `PRETRAINED = False`: every layer starts from torchvision's random initialisation. It needs `RESNET_MODE=finetune`. These runs go to `WaveletNL/table2_resnet18_runs_fixednorm_scratch/finetune/` (a run folder holds one kind of run, and the notebook stops when it finds the other kind) and are compared with the runs of 04 from random weights (`table2_runs_fixednorm_scratch/`).

> **Table 2** is Table 2 of the original submission (the DHS experiment of the preliminary study). Its rerun with the nighttime-light target is reported in Supplementary Section S4 of the revised paper.

# Import libraries

In [ ]:
import os
import json
import time
import datetime
import multiprocessing

# Same as in 01-03: the GPU image sets PROJ_LIB to a folder without proj.db, which breaks rasterio's CRS handling.
for _var in ('PROJ_LIB', 'PROJ_DATA'):
    if os.environ.get(_var) and not os.path.exists(os.path.join(os.environ[_var], 'proj.db')):
        os.environ.pop(_var)

import numpy as np
import pandas as pd
import pywt
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
try:
    from tqdm.auto import tqdm
except ImportError:                      # progress bars are optional
    def tqdm(it, **kw):
        return it

# Settings

In [ ]:
# ---- model ----------------------------------------------------------------------------------------
RESNET_MODE = os.environ.get('RESNET_MODE', 'finetune')   # 'finetune': the first convolution takes all bands and all
                                                           # layers learn; 'frozen': the setup of 04 with ResNet-18

# ---- input / output ------------------------------------------------------------------------
DATA_DIR      = '/path/to/data/'   # the IWI study: Dataset/, its
                                                                    # folds (03) and the band caches of its 04-15
WORK_DIR      = os.path.join(DATA_DIR, 'WaveletNL')                 # this study: the folds file with the target, runs
FOLDS_CSV     = os.path.join(WORK_DIR, 'train_clusters_folds_nl.csv')   # 03_Folds_NL_Target: the folds + the target
IWI_FOLDS_CSV = os.path.join(DATA_DIR, 'train_clusters_folds.csv')  # 03 of the IWI study: FOLDS_CSV lists its clusters
BAND_CACHE    = os.path.join(DATA_DIR, 'table1_cache')              # Landsat, NDVI and wavelet bands, ~2.5 MB per cluster
RUN_ROOT      = os.path.join(WORK_DIR, 'table2_resnet18_runs_fixednorm', RESNET_MODE)   # one folder per row and fold
VGG16_RUNS    = os.path.join(WORK_DIR, 'table2_runs_fixednorm')    # the runs of 04, for the comparison with VGG-16
TORCH_HOME    = '/path/to/torch_home'   # ResNet-18 ImageNet weights

# ---- data (as in 04, so that the caches are shared) --------------------------------------------------
TARGET       = 'nl'        # the NL target of 02: the mean of the NL image of the patch, clipped to NL_CLIP and
                           # normalised with NormFunc of the survey year (a column of FOLDS_CSV)
NL_SENSOR    = 'VIIRS'     # the paper used Landsat 8 and VIIRS (2014 and later); 'DMSP' runs the clusters up to 2013
NDVI_SERIES  = 'monthly'   # 'monthly': the 23 MODIS composites averaged to 12 months, the paper's 12 values;
                           # '16day': all 23 composites as the series
WAVELET      = 'dmey'      # discrete Meyer, filter length 62, as in the paper
WAVELET_MODE = 'symmetric' # pywt's default signal extension
NODATA       = -999999.0   # no-data value of the 01 files; clusters with any no-data pixel are left out

# ---- normalisation of the inputs (fixed numbers, no statistics of the data) ------------------------------------
# Landsat: the Collection 2 Level 2 scale factors of the GEE catalog (LANDSAT/LC08/C02/T1_L2; the same for Landsat 5,
# 7 and 9): SR_B* (BLUE ... SWIR2) DN * 0.0000275 - 0.2 = surface reflectance; ST_B10 / ST_B6 (TEMP1)
# DN * 0.00341802 + 149.0 = surface temperature in kelvin, divided by TEMP1_DIVISOR. NDVI and wavelet bands are used
# as they are. The nighttime light is not an input: it is the target. Nighttime light: clipped to NL_CLIP, then
# (NL - Mean) / stdDev with NormFunc of the survey year, averaged over the patch (02). NL_CLIP and NormFunc are those
# of 02; here they only check the target (section 3).
TEMP1_DIVISOR = 300.0      # kelvin / 300 as in the paper's Keras notebooks (1.0: kelvin, about 300, as in GEE)
NL_CLIP = (0, 150)         # as in 26_tfrecords_generation_DMSP_VIIRS (VIIRS max 150, DMSP max 63); None: no clipping


def NormFunc(Year):
    if (Year >= 2014):     # VIIRS
        Mean = 0.1326654328849849
        stdDev = 1.7479698750873318
    else:                  # DMSP
        Mean = 4.152879625271373
        stdDev = 3.4762732210548317

    return Mean, stdDev


# ---- training, as in 04 -------------------------------------------------------------------------------
FOLDS       = ['A', 'B', 'C', 'D', 'E']
EPOCHS      = 100
BATCH_SIZE  = 64
LR          = 1e-3
L2_HEAD     = 0.01         # Keras l2(0.01) on the output weights: loss + 0.01 * sum(w ** 2)
SEED        = 3127
PRETRAINED  = True         # start from the ImageNet weights (False: random start, like the DHS_30 notebooks)
AMP         = True         # bfloat16 autocast on the GPU
GPU_DATA_GB = 60           # a row's bands go into GPU memory when they need less than this, otherwise they are read
                           # from the caches for every batch (slower)
N_WORKERS   = len(os.sched_getaffinity(0))   # processes that build the caches if they are missing

# ---- what to run: 'all', 'none' (only the caches and the table) or keys of TABLE2, comma-separated.
# These can also be set in the environment.
RUN_EXPERIMENTS = os.environ.get('TABLE2_EXPERIMENTS', 'all')
RUN_FOLDS       = os.environ.get('TABLE2_FOLDS', ','.join(FOLDS)).split(',')

# ---- TRAIN_FROM_SCRATCH=1 in the environment is the same as PRETRAINED = False and
# needs RESNET_MODE=finetune. These runs go to their own folder, table2_resnet18_runs_fixednorm_scratch/<mode>/: a
# run folder holds one kind of run (check_start() stops the notebook otherwise).
_scratch = os.environ.get('TRAIN_FROM_SCRATCH', '0').strip()
assert _scratch in ('', '0', '1'), f'TRAIN_FROM_SCRATCH must be 0 or 1, not {_scratch!r}'
if _scratch == '1':
    assert RESNET_MODE == 'finetune', 'TRAIN_FROM_SCRATCH=1 needs RESNET_MODE=finetune (frozen keeps the layers fixed)'
    PRETRAINED = False
TRAIN_FROM_SCRATCH = not PRETRAINED   # written to result.json and the checkpoints
if TRAIN_FROM_SCRATCH:                # their own run folder (16-18 read both), compared with 04 from random weights
    RUN_ROOT = os.path.join(WORK_DIR, 'table2_resnet18_runs_fixednorm_scratch', RESNET_MODE)
    VGG16_RUNS += '_scratch'
START_NOTE = ', from random weights' if TRAIN_FROM_SCRATCH else ''   # for the table and the plot

os.environ['TORCH_HOME'] = TORCH_HOME
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(no GPU: only for code tests)')
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| pywt', pywt.__version__,
      '| cache workers:', N_WORKERS)
print('ResNet-18 mode:', RESNET_MODE, '| ImageNet start:', PRETRAINED, '| target:', TARGET, '| runs ->', RUN_ROOT)

# 1. The rows of Table 2 without NL

Every row starts with the base model: the 7 Landsat bands (Table 2's base model without NL, which is the target here). The 6 wavelet bands are in the order of bands 12–17 of the TF records: LFC minimum, maximum, mean, then HFC minimum, maximum, mean. The keys are those of Table 2 (04 and 05 of the IWI study), so the runs of both studies can be compared row by row.

In [ ]:
LANDSAT = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'TEMP1']
N_NDVI = 12 if NDVI_SERIES == 'monthly' else 23
NDVI = [f'NDVI_{k:02d}' for k in range(1, N_NDVI + 1)]
WAVELET_BANDS = ['LFC_min', 'LFC_max', 'LFC_mean', 'HFC_min', 'HFC_max', 'HFC_mean']
CHANNELS = LANDSAT + NDVI + WAVELET_BANDS          # the bands of the band cache, in this order
BASE = LANDSAT                                     # the base model of Table 2 without NL (NL is the target)

# key: (Table 2 row without NL, input bands)
TABLE2 = {
    'base':          ('Base model using 7 raw Landsat images', BASE),
    'base_ndvi':     ('Base model with temporal NDVI', BASE + NDVI),
    'base_wavelet6': ('Base model and 6 statistical wavelet-based features', BASE + WAVELET_BANDS),
    'base_hfc_max':  ('Base model and max values of HFC', BASE + ['HFC_max']),
    'base_hfc_mean': ('Base model and mean values of HFC', BASE + ['HFC_mean']),
    'base_hfc_min':  ('Base model and min values of HFC', BASE + ['HFC_min']),
    'base_lfc_max':  ('Base model and max values of LFC', BASE + ['LFC_max']),
    'base_lfc_mean': ('Base model and mean values of LFC', BASE + ['LFC_mean']),
    'base_lfc_min':  ('Base model and min values of LFC', BASE + ['LFC_min']),
}

if RUN_EXPERIMENTS == 'all':
    todo = list(TABLE2)
elif RUN_EXPERIMENTS == 'none':
    todo = []
else:
    todo = [k.strip() for k in RUN_EXPERIMENTS.split(',') if k.strip()]
bad = [k for k in todo if k not in TABLE2]
assert not bad, f'unknown rows: {bad}'
assert set(RUN_FOLDS) <= set(FOLDS), RUN_FOLDS
print('rows to run:', todo, '| test folds:', RUN_FOLDS)
pd.DataFrame([(k, d, len(b), ', '.join(b)) for k, (d, b) in TABLE2.items()],
             columns=['key', 'Table 2 row without NL', 'bands', 'input bands'])

# 2. Clusters

All clusters of `train_clusters_folds_nl.csv` (03: the folds of the IWI study with the NL target of 02) whose nighttime light comes from `NL_SENSOR`, in the order of the band cache (sorted by `cluster_id`). The file must list the clusters of `train_clusters_folds.csv` of the IWI study, with its folds: the caches of the IWI study hold the clusters in this order. They keep the fold of 03. Clusters without a target value are left out of the training only.

In [ ]:
clusters = pd.read_csv(FOLDS_CSV, dtype={'cluster_id': str})
iwi_folds = pd.read_csv(IWI_FOLDS_CSV, usecols=['cluster_id', 'fold'], dtype={'cluster_id': str, 'fold': str})
if (clusters['cluster_id'].tolist() != iwi_folds['cluster_id'].tolist()
        or clusters['fold'].fillna('').astype(str).tolist() != iwi_folds['fold'].fillna('').tolist()):
    raise RuntimeError(f'{FOLDS_CSV} does not list the clusters and folds of {IWI_FOLDS_CSV}: the folds of the IWI study '
                       'changed. Run 02_NL_Target and 03_Folds_NL_Target again (and train in new run folders).')
if 'nl_sensor' not in clusters:                    # 01 uses DMSP up to 2013 and VIIRS from 2014
    clusters['nl_sensor'] = np.where(clusters['year'] >= 2014, 'VIIRS', 'DMSP')
if 'folder' not in clusters:
    clusters['folder'] = [os.path.join(DATA_DIR, 'Dataset', str(c).replace('/', '_')) for c in clusters['cluster_id']]
clusters = clusters[clusters['nl_sensor'] == NL_SENSOR].sort_values('cluster_id').reset_index(drop=True)
assert TARGET in clusters, f'{FOLDS_CSV} has no column {TARGET}'
has_target = clusters[TARGET].notna().to_numpy()
print(f'{len(clusters)} clusters with {NL_SENSOR} nighttime light, {has_target.sum()} with {TARGET}, '
      f'{clusters["country"].nunique()} countries, years {clusters["year"].min()}-{clusters["year"].max()}')
print(f'{TARGET}:', clusters.loc[has_target, TARGET].describe().round(2).to_dict(), '\n')
print(pd.crosstab(clusters['country'], clusters['fold'], margins=True).to_string())

# 3. Band caches

Every cluster's bands are computed once from its three GeoTIFFs of 01 and stored as float16 in the band cache of the IWI study (`table1_cache/`, about 2.5 MB per cluster), shared by 04 to 15 of both studies: the 7 Landsat bands with the GEE scale factors, surface reflectance (DN × 0.0000275 − 0.2) and TEMP1 in kelvin / 300 ((DN × 0.00341802 + 149) / 300); the NDVI series (for `monthly`, each MODIS composite counts for the month in which it starts, and the composites of a month are averaged); and the 6 wavelet bands of that series. The cache also checks the `NL.tif` (a cluster with a no-data pixel in any of its three files is left out), but the NL image is not an input: the target comes from `FOLDS_CSV`.

The rest of the fixed normalisation of Settings is applied when the bands are loaded for training (`normalise()`: TEMP1 is rescaled if `TEMP1_DIVISOR` is not 300). The table at the end of the next cell shows the resulting training values for a sample of clusters.

A cache whose cluster list differs from the table above is built again. A job that finds a cache being built by another job waits for it.

In [ ]:
PATCH = 224
BANDS_DIR = os.path.join(BAND_CACHE, f'{NL_SENSOR}_{NDVI_SERIES}_{WAVELET}_{WAVELET_MODE}')


def month_of_composites(year):
    """Calendar month (1-12) in which each of the 23 MOD13Q1 composites of a year starts (DOY 1, 17, ..., 353)."""
    jan1 = datetime.date(int(year), 1, 1)
    return np.array([(jan1 + datetime.timedelta(days=16 * k)).month for k in range(23)])


def cluster_bands(folder, year):
    """The bands of the band cache for one cluster (float32, CHANNELS order) and its mean NL, or None if a file has
    no-data pixels. TEMP1 is kelvin / 300 here; normalise() applies TEMP1_DIVISOR."""
    with rasterio.open(os.path.join(folder, 'Landsat.tif')) as src:
        ls = src.read().astype(np.float32)
    with rasterio.open(os.path.join(folder, 'MODIS_NDVI.tif')) as src:
        nd = src.read().astype(np.float32)
    with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
        nl = src.read(1).astype(np.float64)
    if (ls == NODATA).any() or (nd == NODATA).any() or (nl == NODATA).any():
        return None
    ls[:6] = ls[:6] * 0.0000275 - 0.2                     # surface reflectance
    ls[6] = (ls[6] * 0.00341802 + 149.0) / 300.0          # kelvin / 300
    if NDVI_SERIES == 'monthly':
        month = month_of_composites(year)
        nd = np.stack([nd[month == m].mean(axis=0) for m in range(1, 13)])
    lfc, hfc = pywt.dwt(nd, WAVELET, mode=WAVELET_MODE, axis=0)    # single-level DWT along time, every pixel
    wav = np.stack([lfc.min(0), lfc.max(0), lfc.mean(0), hfc.min(0), hfc.max(0), hfc.mean(0)])
    return np.concatenate([ls, nd, wav]).astype(np.float32), float(nl.mean())


def cluster_nl(folder, year):
    """The NL band of one cluster (float32, shape (1, 224, 224)) and its mean, or None if it has no-data pixels."""
    with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
        nl = src.read(1).astype(np.float64)
    if (nl == NODATA).any():
        return None
    return nl[None].astype(np.float32), float(nl.mean())


COMPUTE = {'bands': cluster_bands, 'nl': cluster_nl}


def _cache_one(task):
    """Compute one cluster and write it at its place in the .part file. Returns the mean NL and the band sums."""
    i, folder, year, offset, kind, part = task
    try:
        out = COMPUTE[kind](folder, year)
    except Exception as e:
        return i, np.nan, None, f'{type(e).__name__}: {e}'[:200]
    if out is None:
        return i, np.nan, None, 'no-data pixels'
    x, nl_mean = out
    fd = os.open(part, os.O_WRONLY)
    try:
        os.pwrite(fd, x.astype(np.float16).tobytes(), offset)
    finally:
        os.close(fd)
    x64 = x.astype(np.float64)
    return i, nl_mean, np.stack([x64.sum(axis=(1, 2)), (x64 ** 2).sum(axis=(1, 2))]), ''


def build_cache(cache_dir, channels, kind):
    """features.npy, band_stats.csv and index.csv of one cache folder."""
    t0 = time.time()
    features = os.path.join(cache_dir, 'features.npy')
    shape = (len(clusters), len(channels), PATCH, PATCH)
    arr = np.lib.format.open_memmap(features + '.part', mode='w+', dtype=np.float16, shape=shape)
    header, per_cluster = arr.offset, int(np.prod(shape[1:])) * 2
    del arr
    tasks = [(i, r.folder, int(r.year), header + i * per_cluster, kind, features + '.part')
             for i, r in enumerate(clusters.itertuples())]
    nl_mean = np.full(len(clusters), np.nan)
    sums = np.zeros((2, len(channels)))
    error = [''] * len(clusters)
    with multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
        for i, t, s, err in tqdm(pool.imap_unordered(_cache_one, tasks, chunksize=4), total=len(tasks), desc=kind):
            nl_mean[i], error[i] = t, err
            if s is not None:
                sums += s
    ok = np.isfinite(nl_mean)
    n_pix = ok.sum() * PATCH * PATCH
    mean = sums[0] / n_pix
    std = np.sqrt(np.maximum(sums[1] / n_pix - mean ** 2, 1e-12))
    pd.DataFrame({'band': channels, 'mean': mean, 'std': std}).to_csv(os.path.join(cache_dir, 'band_stats.csv'),
                                                                      index=False)
    os.replace(features + '.part', features)
    index = clusters[['cluster_id', 'country', 'year', 'lat', 'lon']].copy()
    index['nl_mean'], index['ok'], index['error'] = nl_mean, ok, error
    index.to_csv(os.path.join(cache_dir, 'index.csv'), index=False)    # the cache counts as complete once this exists
    print(f'{kind} cache: {ok.sum()} of {len(ok)} clusters in {time.time() - t0:.0f} s -> {features}')


def cache_is_complete(cache_dir):
    index, features = os.path.join(cache_dir, 'index.csv'), os.path.join(cache_dir, 'features.npy')
    if not (os.path.exists(index) and os.path.exists(features)):
        return False
    return pd.read_csv(index)['cluster_id'].tolist() == clusters['cluster_id'].tolist()


def ensure_cache(cache_dir, channels, kind):
    """Build a cache folder unless it is complete. Waits when another job is building it (same lock in 04 to 15)."""
    os.makedirs(cache_dir, exist_ok=True)
    lock = os.path.join(cache_dir, 'building.lock')
    if cache_is_complete(cache_dir):
        return
    try:
        os.mkdir(lock)
    except FileExistsError:
        print(f'another job is building {cache_dir} (if none is running, delete {lock}); waiting ...')
        t_wait = time.time()
        while os.path.exists(lock) and time.time() - t_wait < 4 * 3600:
            time.sleep(30)
        assert cache_is_complete(cache_dir), 'the cache is still not complete: start the notebook again'
        return
    try:
        if not cache_is_complete(cache_dir):           # another job may have finished it just now
            build_cache(cache_dir, channels, kind)
    finally:
        os.rmdir(lock)


def load_cache(cache_dir, channels):
    index = pd.read_csv(os.path.join(cache_dir, 'index.csv'))
    feat = np.load(os.path.join(cache_dir, 'features.npy'), mmap_mode='r')
    assert feat.shape[1:] == (len(channels), PATCH, PATCH), feat.shape
    return index, feat


# ---- the fixed normalisation of the cached bands (Settings) -----------------------------------------------------
TEMP_FROM_CACHE = 300.0 / TEMP1_DIVISOR    # the band cache keeps TEMP1 in kelvin / 300


def normalise(x, bands):
    """Training values of cached bands, in place. x: float32 tensor (clusters, bands, 224, 224). Landsat with the GEE
    scale factors (cached as they are, TEMP1 in kelvin / TEMP1_DIVISOR), the rest unchanged."""
    for k, b in enumerate(bands):
        if b == 'TEMP1' and TEMP_FROM_CACHE != 1.0:
            x[:, k] *= TEMP_FROM_CACHE
    return x


ensure_cache(BANDS_DIR, CHANNELS, 'bands')
band_index, FEAT = load_cache(BANDS_DIR, CHANNELS)
complete = band_index['ok'].to_numpy()                    # (the band cache checks NL.tif too)
print(f'band cache {FEAT.shape}, {FEAT.nbytes / 1e9:.1f} GB -> {BANDS_DIR}')
print(f'clusters with complete data: {complete.sum()} of {len(complete)}; left out:',
      band_index.loc[~complete, 'error'].value_counts().to_dict())

# training values of every band for a sample of clusters
sample = np.sort(np.random.default_rng(0).choice(np.flatnonzero(complete), size=min(32, complete.sum()), replace=False))
x = normalise(torch.from_numpy(FEAT[sample].astype(np.float32)), CHANNELS)
print(f'\ntraining values of the input bands, {len(sample)} clusters:')
pd.DataFrame({'min': x.amin(dim=(0, 2, 3)).numpy(), 'mean': x.mean(dim=(0, 2, 3)).numpy(),
              'max': x.amax(dim=(0, 2, 3)).numpy()}, index=CHANNELS).T.round(3)

In [ ]:
# check one cluster against a fresh computation from its files (the cache, and its target against its NL.tif), then
# show its bands and its NL image
usable = np.flatnonzero(complete & has_target)
assert len(usable), f'no cluster has complete data and a {TARGET} value'
i = int(usable[np.argmax(clusters[TARGET].to_numpy()[usable])])      # the brightest cluster (largest target)
row = clusters.iloc[i]
x_new, t_new = cluster_bands(row['folder'], row['year'])
nl_new, _ = cluster_nl(row['folder'], row['year'])
assert np.allclose(FEAT[i].astype(np.float32), x_new, rtol=2e-3, atol=2e-3) and np.isclose(t_new, band_index.at[i, 'nl_mean'])
m_nl, sd_nl = NormFunc(int(row['year']))
nl_img = nl_new[0].astype(np.float64)
target_new = float((((np.clip(nl_img, *NL_CLIP) if NL_CLIP else nl_img) - m_nl) / sd_nl).mean())
assert np.isclose(target_new, row[TARGET], rtol=1e-5, atol=1e-4), (target_new, row[TARGET])
print('cache and target match the files for', row['cluster_id'])

x = FEAT[i].astype(np.float32)
fig, ax = plt.subplots(3, 3, figsize=(12, 12), constrained_layout=True)
rgb = np.clip(x[[2, 1, 0]] / 0.3, 0, 1)
ax[0, 0].imshow(np.moveaxis(rgb, 0, -1))
ax[0, 0].set_title('Landsat RGB')
ax[0, 1].imshow(nl_new[0], cmap='magma')
ax[0, 1].set_title(f'{NL_SENSOR} nighttime light (the target), mean {t_new:.2f}')
nd = x[len(LANDSAT):len(LANDSAT) + N_NDVI]
ax[0, 2].plot(range(1, N_NDVI + 1), nd.reshape(N_NDVI, -1).mean(axis=1), 'o-', color='#2a78d6')
ax[0, 2].set_title('NDVI series, patch mean')
ax[0, 2].set_xlabel('month' if NDVI_SERIES == 'monthly' else '16-day composite')
for k, name in enumerate(WAVELET_BANDS):
    a = ax[1 + k // 3, k % 3]
    a.imshow(x[CHANNELS.index(name)], cmap='viridis')
    a.set_title(name)
for a in ax.flat[[0, 1, 3, 4, 5, 6, 7, 8]]:
    a.set_xticks([])
    a.set_yticks([])
fig.suptitle(f"{row['cluster_id']} ({row['year']}), {TARGET} {row[TARGET]:.2f}")
plt.show()

# 4. Model and training

`ResNet18Regressor` is torchvision's ResNet-18 with one linear output, in the `RESNET_MODE` above. In both modes the output layer has L2 0.01 on its weights and runs in float32. In `finetune` mode a checkpoint holds all weights, including the BatchNorm statistics (about 45 MB). In `frozen` mode it holds only the 1 × 1 convolution and the output layer.

A run trains one row of the table with one test fold. After every epoch it writes the history, the best weights (`best.pt`, lowest validation loss including the L2 term, like Keras `val_loss`) and its full state (`last.pt`). The last step tests `best.pt` on the test fold and writes `test_predictions.csv` and `result.json`, then deletes `last.pt`, which is only needed to continue a stopped run. A run with a `result.json` is skipped. A run with only a `last.pt` continues from that epoch.

If loading the ImageNet weights fails because the machine has no internet access, download them once on a machine that has it, then run again:

```bash
mkdir -p /path/to/torch_home/hub/checkpoints
wget -P /path/to/torch_home/hub/checkpoints \
     https://download.pytorch.org/models/resnet18-f37072fd.pth
```

`Bands` holds the input bands of one row with the fixed normalisation of Settings (`normalise()`, section 3), as in 04. They come from the band cache, in the order of the row: the 7 Landsat bands, then the extra bands.

In [ ]:
RESNET_STATE = None
if PRETRAINED:
    try:
        RESNET_STATE = torchvision.models.resnet18(weights='IMAGENET1K_V1').state_dict()
    except Exception as e:
        raise RuntimeError(f'ResNet-18 ImageNet weights could not be loaded ({type(e).__name__}: {e}). '
                           'Download them as shown above.') from e
RGB_INDEX = {'RED': 0, 'GREEN': 1, 'BLUE': 2}            # channel order of the ImageNet weights


def first_conv_weights(bands, rgb_weights):
    """First-layer weights for any bands from the ImageNet ones (Yeh et al. 2020): RED, GREEN and BLUE keep their
    own, every other band gets their mean, and all are scaled by 3 / number of bands."""
    mean = rgb_weights.mean(dim=1)
    w = torch.stack([rgb_weights[:, RGB_INDEX[b]] if b in RGB_INDEX else mean for b in bands], dim=1)
    return w * 3.0 / len(bands)


class ResNet18Regressor(nn.Module):
    """ResNet-18 with one linear output.
    finetune: the first convolution takes all bands, every layer learns, global average pooling -> 512 features.
    frozen:   1x1 conv (bands -> 3, ReLU), frozen ImageNet convolution layers, flatten -> 7 x 7 x 512 features."""

    def __init__(self, bands, mode):
        super().__init__()
        assert mode in ('finetune', 'frozen'), f'RESNET_MODE must be finetune or frozen, not {mode}'
        self.mode = mode
        net = torchvision.models.resnet18(weights=None)
        if RESNET_STATE is not None:
            net.load_state_dict(RESNET_STATE)
        if mode == 'finetune':
            rgb_weights = net.conv1.weight.detach().clone()
            net.conv1 = nn.Conv2d(len(bands), 64, kernel_size=7, stride=2, padding=3, bias=False)
            with torch.no_grad():
                if RESNET_STATE is not None:
                    net.conv1.weight.copy_(first_conv_weights(bands, rgb_weights))
                else:                                     # torchvision's initialisation of its convolutions
                    nn.init.kaiming_normal_(net.conv1.weight, mode='fan_out', nonlinearity='relu')
            net.fc = nn.Identity()                        # the network now ends with the 512 pooled features
            self.adapter = nn.Identity()
            self.backbone = net
            n_features = 512
        else:
            self.adapter = nn.Sequential(nn.Conv2d(len(bands), 3, kernel_size=1), nn.ReLU())
            nn.init.xavier_uniform_(self.adapter[0].weight)       # Keras defaults, as in 04 and 05
            nn.init.zeros_(self.adapter[0].bias)
            self.backbone = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool,
                                          net.layer1, net.layer2, net.layer3, net.layer4, nn.Flatten())
            for p in self.backbone.parameters():
                p.requires_grad_(False)
            n_features = 512 * 7 * 7
        self.head = nn.Linear(n_features, 1)
        nn.init.xavier_uniform_(self.head.weight)                 # Keras defaults: Glorot uniform, zero bias
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        x = self.backbone(self.adapter(x))
        with torch.autocast(device_type=x.device.type, enabled=False):    # output layer in float32
            return self.head(x.float()).squeeze(1)

    def train(self, mode=True):
        super().train(mode)
        if self.mode == 'frozen':
            self.backbone.eval()                  # the frozen BatchNorm layers keep their ImageNet statistics
        return self

    def learned_state(self):
        """What a checkpoint keeps: every weight and BatchNorm statistic for finetune, the 1x1 convolution and the
        output layer for frozen."""
        if self.mode == 'finetune':
            return self.state_dict()
        return {'adapter': self.adapter.state_dict(), 'head': self.head.state_dict()}

    def load_learned_state(self, state):
        if self.mode == 'finetune':
            self.load_state_dict(state)
        else:
            self.adapter.load_state_dict(state['adapter'])
            self.head.load_state_dict(state['head'])


def check_start(from_scratch, where):
    """A run folder holds one kind of run: stop instead of mixing runs from pretrained and from random weights
    (TRAIN_FROM_SCRATCH). Runs written before 2026-11-09 have no 'from_scratch' and started from pretrained weights."""
    if bool(from_scratch) != TRAIN_FROM_SCRATCH:
        kind = {True: 'from random weights', False: 'from the pretrained weights'}
        raise RuntimeError(f'{where} holds a run {kind[bool(from_scratch)]}, but this run starts '
                           f'{kind[TRAIN_FROM_SCRATCH]}: give it another RUN_ROOT')


SOURCE = {b: ('bands', CHANNELS.index(b)) for b in CHANNELS}            # where every input band is cached


class Bands:
    """The input bands of one row for all clusters, from the band cache, with the fixed normalisation of normalise().
    The cached values go into DEVICE memory when they take less than GPU_DATA_GB, otherwise every batch is read from
    the caches."""

    def __init__(self, bands, rows):
        self.bands = list(bands)
        self.n_bands = len(self.bands)
        self.rows = np.asarray(rows)
        self.ch = np.array([SOURCE[b][1] for b in self.bands], dtype=int)   # their channels in the band cache
        gb = len(self.rows) * self.n_bands * PATCH * PATCH * 2 / 1e9
        self.x = None
        if gb < GPU_DATA_GB:
            self.x = torch.empty((len(self.rows), self.n_bands, PATCH, PATCH), dtype=torch.float16, device=DEVICE)
            for s in tqdm(range(0, len(self.rows), 256), desc=f'loading {self.n_bands} bands, {gb:.1f} GB', leave=False):
                self.x[s:s + 256] = torch.from_numpy(self.read(self.rows[s:s + 256])).to(DEVICE)

    def read(self, pos):
        """float16 cached values of the cache rows pos, in the order of the row."""
        return np.ascontiguousarray(FEAT[pos[:, None], self.ch[None, :]])

    def batch(self, idx):
        """Normalised float32 bands of the clusters at positions idx (a 1-D tensor)."""
        if self.x is not None:
            x = self.x[idx.to(self.x.device)]
        else:
            x = torch.from_numpy(self.read(self.rows[idx.cpu().numpy()])).to(DEVICE, non_blocking=True)
        x = normalise(x.float(), self.bands)
        return x.contiguous(memory_format=torch.channels_last)


def r2_score(y, p):
    y, p = np.asarray(y, np.float64), np.asarray(p, np.float64)
    return float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP and DEVICE.type == 'cuda')


def predict(model, data, idx, batch=256):
    model.eval()
    out = []
    with torch.no_grad(), autocast():
        for s in range(0, len(idx), batch):
            out.append(model(data.batch(idx[s:s + batch])).float())
    return torch.cat(out)


def save(obj, path):
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)

In [ ]:
def run_fold(key, data, y, folds, test_fold):
    """Train one row of Table 2 with one test fold; returns the test metrics (also in result.json)."""
    run_dir = os.path.join(RUN_ROOT, key, f'test_{test_fold}')
    result_file = os.path.join(run_dir, 'result.json')
    if os.path.exists(result_file):
        with open(result_file) as f:
            result = json.load(f)
        check_start(result.get('from_scratch', False), run_dir)
        return result
    os.makedirs(run_dir, exist_ok=True)
    k = FOLDS.index(test_fold)
    val_fold = FOLDS[(k - 1) % len(FOLDS)]
    pos = lambda mask: torch.from_numpy(np.flatnonzero(mask)).to(DEVICE)
    tr, va, te = pos(~np.isin(folds, [test_fold, val_fold])), pos(folds == val_fold), pos(folds == test_fold)

    torch.manual_seed(SEED + k)
    model = ResNet18Regressor(data.bands, RESNET_MODE).to(DEVICE).to(memory_format=torch.channels_last)
    with torch.no_grad():
        model.head.bias.fill_(float(y[tr].mean()))
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR, eps=1e-7)   # Keras epsilon
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(SEED * 100 + k)
    start, best, best_epoch = 0, float('inf'), -1
    last_file, best_file, hist_file = (os.path.join(run_dir, f) for f in ('last.pt', 'best.pt', 'history.csv'))
    if os.path.exists(last_file):                       # continue a run that was stopped
        st = torch.load(last_file, map_location=DEVICE)
        check_start(st.get('from_scratch', False), run_dir)
        model.load_learned_state(st['model'])
        opt.load_state_dict(st['opt'])
        gen.set_state(st['gen'].cpu())
        start, best, best_epoch = st['epoch'] + 1, st['best'], st['best_epoch']
        print(f'{key} test {test_fold}: continuing at epoch {start + 1}')

    for epoch in range(start, EPOCHS):
        t0 = time.time()
        model.train()
        perm = tr[torch.randperm(len(tr), generator=gen, device=DEVICE)]
        sq_sum = torch.zeros((), device=DEVICE)
        n_seen = 0
        for s in range(0, len(perm) - BATCH_SIZE + 1, BATCH_SIZE):     # last incomplete batch dropped, as in TF
            b = perm[s:s + BATCH_SIZE]
            with autocast():
                pred = model(data.batch(b))
            mse = F.mse_loss(pred.float(), y[b])
            loss = mse + L2_HEAD * model.head.weight.pow(2).sum()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sq_sum += mse.detach() * len(b)
            n_seen += len(b)
        reg = L2_HEAD * float(model.head.weight.detach().pow(2).sum())
        p_va = predict(model, data, va).cpu().numpy()
        y_va = y[va].cpu().numpy()
        val_mse = float(((p_va - y_va) ** 2).mean())
        val_loss = val_mse + reg
        if val_loss < best:
            best, best_epoch = val_loss, epoch
            save({'model': model.learned_state(), 'epoch': epoch, 'from_scratch': TRAIN_FROM_SCRATCH}, best_file)
        rec = {'epoch': epoch + 1, 'train_mse': float(sq_sum) / max(n_seen, 1), 'l2_term': reg, 'val_mse': val_mse,
               'val_loss': val_loss, 'val_r2': r2_score(y_va, p_va), 'seconds': round(time.time() - t0, 1)}
        pd.DataFrame([rec]).to_csv(hist_file, mode='a', header=not os.path.exists(hist_file), index=False)
        save({'model': model.learned_state(), 'opt': opt.state_dict(), 'gen': gen.get_state(), 'epoch': epoch,
              'best': best, 'best_epoch': best_epoch, 'from_scratch': TRAIN_FROM_SCRATCH}, last_file)
        if epoch == start or (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
            print(f"{key} test {test_fold} epoch {epoch + 1}/{EPOCHS}: train mse {rec['train_mse']:.3f}, "
                  f"val mse {val_mse:.3f}, val R2 {rec['val_r2']:.3f}, best epoch {best_epoch + 1}, {rec['seconds']} s")

    st = torch.load(best_file, map_location=DEVICE)
    check_start(st.get('from_scratch', False), run_dir)
    model.load_learned_state(st['model'])
    p_te = predict(model, data, te).cpu().numpy()
    y_te = y[te].cpu().numpy()
    test_rows = clusters.iloc[data.rows[te.cpu().numpy()]]
    pd.DataFrame({'cluster_id': test_rows['cluster_id'].to_numpy(), 'country': test_rows['country'].to_numpy(),
                  'year': test_rows['year'].to_numpy(), 'y_true': y_te, 'y_pred': p_te}
                 ).to_csv(os.path.join(run_dir, 'test_predictions.csv'), index=False)
    result = {'key': key, 'model': f'resnet18_{RESNET_MODE}', 'test_fold': test_fold, 'val_fold': val_fold,
              'bands': data.n_bands, 'n_train': len(tr), 'n_val': len(va), 'n_test': len(te),
              'best_epoch': st['epoch'] + 1, 'test_r2': r2_score(y_te, p_te),
              'test_mse': float(((p_te - y_te) ** 2).mean()), 'test_mae': float(np.abs(p_te - y_te).mean()),
              'epochs': EPOCHS, 'from_scratch': TRAIN_FROM_SCRATCH, 'target': TARGET}
    with open(result_file + '.tmp', 'w') as f:
        json.dump(result, f, indent=1)
    os.replace(result_file + '.tmp', result_file)
    if os.path.exists(last_file):
        os.remove(last_file)                            # only needed to continue a stopped run
    print(f"{key} test {test_fold}: test R2 {result['test_r2']:.3f} (best epoch {result['best_epoch']})")
    return result

# 5. Train

The clusters with complete data and a target value. The bands of a row are loaded once and used for its five test folds. Rows and test folds come from `RUN_EXPERIMENTS` and `RUN_FOLDS`.

In [ ]:
use = complete & has_target
rows = np.flatnonzero(use)                                   # cache rows of the clusters used
folds = clusters['fold'].to_numpy()[use]
y = torch.tensor(clusters[TARGET].to_numpy(dtype=np.float64)[use], dtype=torch.float32, device=DEVICE)
print(f'{len(rows)} clusters:', pd.Series(folds).value_counts().sort_index().to_dict())
if todo and DEVICE.type != 'cuda':
    print('WARNING: no GPU, training will be very slow')

for key in todo:
    desc, bands = TABLE2[key]
    missing = [f for f in RUN_FOLDS if not os.path.exists(os.path.join(RUN_ROOT, key, f'test_{f}', 'result.json'))]
    if not missing:
        print(f'{key}: all test folds done')
        continue
    print(f'\n=== {key}: {desc} ({len(bands)} bands), test folds {missing}')
    data = Bands(bands, rows)
    for f in missing:
        run_fold(key, data, y, folds, f)
    del data
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

# 6. Table 2 without NL with ResNet-18

For every row: the mean and standard deviation of the test R² over the folds that are done, and the R² and RMSE (in units of the target: NormFunc standard deviations) of all test predictions pooled. Next to them is the mean test R² of the VGG-16 runs of 04 (`VGG16_RUNS`). The results go to `table2_resnet18_<mode>_results.csv` and `table2_resnet18_<mode>_per_fold.csv` in `RUN_ROOT`.

In [ ]:
def fold_results(run_root, key):
    """result.json of every finished test fold of a row, and its pooled test predictions (or None)."""
    found, preds = [], []
    for f in FOLDS:
        run_dir = os.path.join(run_root, key, f'test_{f}')
        if os.path.exists(os.path.join(run_dir, 'result.json')):
            with open(os.path.join(run_dir, 'result.json')) as fh:
                found.append(json.load(fh))
            preds.append(pd.read_csv(os.path.join(run_dir, 'test_predictions.csv')))
    return found, (pd.concat(preds) if preds else None)


per_fold, rows_out = [], []
for key, (desc, bands) in TABLE2.items():
    found, p = fold_results(RUN_ROOT, key)
    for x in found:                                       # a run folder holds one kind of run
        check_start(x.get('from_scratch', False), os.path.join(RUN_ROOT, key, f"test_{x['test_fold']}"))
    per_fold += found
    r = pd.Series([x['test_r2'] for x in found], dtype=float)
    vgg = pd.Series([x['test_r2'] for x in fold_results(VGG16_RUNS, key)[0]], dtype=float)
    out = {'key': key, 'Table 2 row without NL': desc, 'bands': len(bands),
           'VGG-16 R2 (04)': vgg.mean() if len(vgg) else np.nan, 'VGG-16 folds': len(vgg), 'folds done': len(r),
           'test R2 mean': r.mean() if len(r) else np.nan, 'test R2 std': r.std() if len(r) > 1 else np.nan,
           'test R2 min': r.min() if len(r) else np.nan, 'test R2 max': r.max() if len(r) else np.nan,
           'pooled R2': r2_score(p['y_true'], p['y_pred']) if p is not None else np.nan}
    out['pooled RMSE'] = float(np.sqrt(((p['y_true'] - p['y_pred']) ** 2).mean())) if p is not None else np.nan
    rows_out.append(out)
per_fold = pd.DataFrame(per_fold, columns=['key', 'model', 'test_fold', 'val_fold', 'bands', 'n_train', 'n_val',
                                           'n_test', 'best_epoch', 'test_r2', 'test_mse', 'test_mae', 'epochs', 'target'])
results = pd.DataFrame(rows_out)
os.makedirs(RUN_ROOT, exist_ok=True)
for frame, name in ((results, f'table2_resnet18_{RESNET_MODE}_results.csv'),
                    (per_fold, f'table2_resnet18_{RESNET_MODE}_per_fold.csv')):
    path = os.path.join(RUN_ROOT, name)
    frame.to_csv(path + '.tmp', index=False)              # parallel runs may write at the same time
    os.replace(path + '.tmp', path)
print(f"ResNet-18 {RESNET_MODE}{START_NOTE}, {TARGET} target, {NL_SENSOR} clusters, NDVI series '{NDVI_SERIES}',"
      f" {len(rows)} clusters, spatial folds of 03; VGG-16 results from {VGG16_RUNS}\n")
results.round(3)

In [ ]:
done = results[results['folds done'] > 0].iloc[::-1]
if len(done):
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(done) + 1.4), constrained_layout=True)
    yy = np.arange(len(done))
    handles = [ax.barh(yy, done['test R2 mean'], xerr=done['test R2 std'].fillna(0), height=0.6, color='#2a78d6',
                       error_kw={'ecolor': '#52514e', 'elinewidth': 1, 'capsize': 2},
                       label=f'ResNet-18 {RESNET_MODE}: mean ± sd of test folds')]
    if done['VGG-16 R2 (04)'].notna().any():
        handles.append(ax.scatter(done['VGG-16 R2 (04)'], yy, marker='o', s=40, color='#1baf7a', edgecolors='white',
                                  linewidths=1.2, zorder=3, label='VGG-16, 04 (mean of test folds)'))
    for k, v, e in zip(yy, done['test R2 mean'], done['test R2 std'].fillna(0)):
        if v > 0.15:                                       # value inside the bar, clear of the markers
            ax.text(0.01, k, f'{v:.2f}', va='center', ha='left', color='white', fontsize=8)
        else:
            ax.text(min(max(v + e, 0) + 0.015, 0.92), k, f'{v:.2f}', va='center', ha='left', color='#52514e',
                    fontsize=8)
    ax.set_yticks(yy)
    ax.set_yticklabels(done['key'])
    ax.set_xlim(min(0, np.nanmin(done[['test R2 mean', 'VGG-16 R2 (04)']].to_numpy(float)) - 0.05), 1)
    ax.set_xlabel('test R²')
    ax.grid(axis='x', color='#e1e0d9', linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.1 - 0.6 / len(done)), ncol=3,
              frameon=False, fontsize=8)
    ax.set_title(f'Table 2 without NL, ResNet-18 ({RESNET_MODE}{START_NOTE}), {TARGET.upper()} target of the DHS clusters, '
                 'spatial 5-fold CV', loc='left', fontsize=11)
    png = os.path.join(RUN_ROOT, f'table2_resnet18_{RESNET_MODE}_r2.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()
else:
    print('no finished runs yet')